# 02b — Normalize Editorial Labels

**PURPOSE**: Extract the leading `[bracket]` token from each article's
`rawTitle` in `article_index`, and separate the immutable raw token from an
explicitly-aliased normalized form. Bracket labels are editorial weak labels,
not a taxonomy ground truth — normalization here is a narrow allowlist
(`config/label_normalization.yaml`), never a blanket transformation.

**INPUT**:
- `data/20_processed/mbn/life/article_index.parquet` (225 rows, from 02a)
- `config/label_normalization.yaml` (alias map + excluded-label list)

**OUTPUT**:
- `data/20_processed/mbn/life/title_labels.{parquet,csv}` (one row per
  article that has a bracket token — 102 rows expected for this cohort)
- `data/80_quality/label_normalization_report.json` (raw vs. normalized
  frequency tables)

**DEPENDENCIES**: pandas, pyarrow, pyyaml, `src.parsing.mbn_index_parser`,
`src.validation.checks`

**PARAMETERS**: `config/label_normalization.yaml:aliases`,
`config/label_normalization.yaml:excluded_editorial_labels`

**ASSUMPTIONS**: `rawBracketToken` extraction re-runs the same leading-bracket
regex already used in 02a (`extract_bracket_label`); each title has at most
one leading bracket token in this dataset (no observed multi-bracket titles),
so `title_label` also happens to satisfy `articleId` uniqueness here even
though the contract permits zero-or-more rows per article in general.

**SIDE EFFECTS**: writes the two output artifacts above; does not modify
`article_index` or `data/00_raw/**`.

**FAIL CONDITIONS**: raises `PrimaryKeyViolation` if any `rawBracketToken` is
null/empty for a row that was emitted (a contradiction — such rows must not
be emitted at all), or if an alias in the config maps a token to itself
(a no-op alias entry, which indicates a config authoring mistake).

In [1]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (.git marker) from {start}")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: /home/sieg/projects-wsl/mbN_GUIDE_PY/mbN_GUIDE


In [2]:
import json
from datetime import datetime, timezone

import pandas as pd
import yaml

from src.io.paths import config_path, data_dir, ensure_parent
from src.io.parquet_io import read_table, write_table
from src.parsing.mbn_index_parser import INDEX_PARSER_VERSION, extract_bracket_label
from src.validation.checks import PrimaryKeyViolation, assert_primary_key, null_counts, duplicate_report

with open(config_path(REPO_ROOT, "label_normalization.yaml"), encoding="utf-8") as f:
    LABEL_CFG = yaml.safe_load(f)

NORMALIZATION_VERSION = LABEL_CFG["version"]
ALIASES = LABEL_CFG["aliases"]
EXCLUDED_LABELS = set(LABEL_CFG["excluded_editorial_labels"])

for raw, normalized in ALIASES.items():
    if raw == normalized:
        raise ValueError(f"No-op alias in config: '{raw}' -> '{normalized}'")

INPUT_INDEX_PATH = data_dir(REPO_ROOT, "20_processed", "mbn", "life", "article_index")
OUTPUT_LABELS_PATH = data_dir(REPO_ROOT, "20_processed", "mbn", "life", "title_labels")
OUTPUT_REPORT_PATH = data_dir(REPO_ROOT, "80_quality", "label_normalization_report.json")

EXECUTION_TIMESTAMP = datetime.now(timezone.utc).isoformat()

article_index = read_table(INPUT_INDEX_PATH)
print("article_index rows:", len(article_index))
print("aliases:", ALIASES)
print("excluded_editorial_labels:", EXCLUDED_LABELS)

article_index rows: 225
aliases: {'AI 기상캐스터': 'AI기상캐스터'}
excluded_editorial_labels: {'반론보도'}


## Extract rawBracketToken, apply alias normalization + exclusion flag

In [3]:
EXTRACTION_METHOD = "regex_leading_bracket"
EXTRACTION_VERSION = INDEX_PARSER_VERSION

rows = []
for _, art in article_index.iterrows():
    raw_token = extract_bracket_label(art["rawTitle"])
    if raw_token is None:
        continue  # title_label has zero rows for unlabeled articles
    normalized_token = ALIASES.get(raw_token, raw_token)
    rows.append(
        {
            "articleId": art["articleId"],
            "rawBracketToken": raw_token,
            "normalizedBracketToken": normalized_token,
            "normalizationMethod": "alias" if raw_token in ALIASES else "identity",
            "normalizationVersion": NORMALIZATION_VERSION,
            "isExcludedEditorialLabel": raw_token in EXCLUDED_LABELS,
            "extractionMethod": EXTRACTION_METHOD,
            "extractionVersion": EXTRACTION_VERSION,
        }
    )

title_labels = pd.DataFrame.from_records(rows)
print("title_labels rows:", len(title_labels))
title_labels.head(5)

title_labels rows: 102


,articleId,rawBracketToken,normalizedBracketToken,normalizationMethod,normalizationVersion,isExcludedEditorialLabel,extractionMethod,extractionVersion
0,5210942,AI기상캐스터,AI기상캐스터,identity,label_normalization@1,False,regex_leading_bracket,mbn_index_parser@1
1,5210752,Season Item,Season Item,identity,label_normalization@1,False,regex_leading_bracket,mbn_index_parser@1
2,5210310,Health Recipe,Health Recipe,identity,label_normalization@1,False,regex_leading_bracket,mbn_index_parser@1
3,5210190,AI 기상캐스터,AI기상캐스터,alias,label_normalization@1,False,regex_leading_bracket,mbn_index_parser@1
4,5209482,Find Dining,Find Dining,identity,label_normalization@1,False,regex_leading_bracket,mbn_index_parser@1


## Integrity checks

In [4]:
empty_tokens = title_labels["rawBracketToken"].isna() | (title_labels["rawBracketToken"].str.len() == 0)
if empty_tokens.any():
    raise PrimaryKeyViolation(f"{int(empty_tokens.sum())} emitted title_label rows have an empty rawBracketToken")

# In this dataset each article has at most one leading-bracket token, so
# title_label also happens to be articleId-unique; this is a dataset property,
# not a schema guarantee (see ASSUMPTIONS above), and we assert it here so a
# future article with two brackets is caught rather than silently overwriting.
assert_primary_key(title_labels, ["articleId"], context="title_labels (dataset-specific 1-bracket assumption)")

checks = {
    "row_count": int(len(title_labels)),
    "expected_row_count": 102,
    "null_counts": null_counts(title_labels),
    "duplicate_counts": duplicate_report(title_labels, [["articleId"]]),
    "excluded_count": int(title_labels["isExcludedEditorialLabel"].sum()),
    "aliased_count": int((title_labels["normalizationMethod"] == "alias").sum()),
}
for k, v in checks.items():
    print(f"{k}: {v}")

row_count: 102
expected_row_count: 102
null_counts: {'articleId': 0, 'rawBracketToken': 0, 'normalizedBracketToken': 0, 'normalizationMethod': 0, 'normalizationVersion': 0, 'isExcludedEditorialLabel': 0, 'extractionMethod': 0, 'extractionVersion': 0}
duplicate_counts: {'articleId': 0}
excluded_count: 1
aliased_count: 1


## Raw vs. normalized label frequency (both reported, neither discarded)

In [5]:
raw_freq = title_labels["rawBracketToken"].value_counts().to_dict()
normalized_freq = title_labels["normalizedBracketToken"].value_counts().to_dict()

print("=== raw label frequency ===")
for label, cnt in sorted(raw_freq.items(), key=lambda x: -x[1]):
    print(f"  [{label}]: {cnt}")

print("\n=== normalized label frequency ===")
for label, cnt in sorted(normalized_freq.items(), key=lambda x: -x[1]):
    print(f"  [{label}]: {cnt}")

print("\nunique raw labels:", len(raw_freq), "-> unique normalized labels:", len(normalized_freq))

=== raw label frequency ===
  [Find Dining]: 24
  [Health Recipe]: 20
  [Style]: 20
  [Consumer News]: 11
  [AI기상캐스터]: 10
  [Mind Note]: 7
  [Issue Pick]: 2
  [Season Item]: 1
  [AI 기상캐스터]: 1
  [Travel Trend]: 1
  [Travel News]: 1
  [반론보도]: 1
  [Hotel News]: 1
  [진료는 의사에게]: 1
  [MBN이 본 신간]: 1

=== normalized label frequency ===
  [Find Dining]: 24
  [Health Recipe]: 20
  [Style]: 20
  [AI기상캐스터]: 11
  [Consumer News]: 11
  [Mind Note]: 7
  [Issue Pick]: 2
  [Season Item]: 1
  [Travel Trend]: 1
  [Travel News]: 1
  [반론보도]: 1
  [Hotel News]: 1
  [진료는 의사에게]: 1
  [MBN이 본 신간]: 1

unique raw labels: 15 -> unique normalized labels: 14


## Write outputs

In [6]:
write_manifest = write_table(title_labels, OUTPUT_LABELS_PATH, required_columns=["articleId", "rawBracketToken"])

quality_status = "PASS" if checks["row_count"] == checks["expected_row_count"] else "WARN"

report = {
    "notebook": "02bNormalizeEditorialLabels.ipynb",
    "executionTimestamp": EXECUTION_TIMESTAMP,
    "normalizationVersion": NORMALIZATION_VERSION,
    "extractionVersion": EXTRACTION_VERSION,
    "aliasesApplied": ALIASES,
    "excludedEditorialLabels": sorted(EXCLUDED_LABELS),
    "checks": checks,
    "rawLabelFrequency": raw_freq,
    "normalizedLabelFrequency": normalized_freq,
    "qualityStatus": quality_status,
    "outputArtifact": write_manifest,
}

ensure_parent(OUTPUT_REPORT_PATH)
with open(OUTPUT_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)

print("qualityStatus:", quality_status)
print("wrote:", OUTPUT_REPORT_PATH.relative_to(REPO_ROOT))

qualityStatus: PASS
wrote: data/80_quality/label_normalization_report.json


## Closing summary

In [7]:
print("=== ROW COUNTS ===")
print({"title_labels": len(title_labels)})

print("=== NULL COUNTS ===")
print(checks["null_counts"])

print("=== DUPLICATES ===")
print(checks["duplicate_counts"])

print("=== QUALITY METRICS ===")
print({
    "row_count_matches_expected": checks["row_count"] == checks["expected_row_count"],
    "excluded_count": checks["excluded_count"],
    "aliased_count": checks["aliased_count"],
    "unique_raw_labels": len(raw_freq),
    "unique_normalized_labels": len(normalized_freq),
})

print("=== OUTPUT PATH ===")
print(write_manifest["parquet_path"], "|", write_manifest["csv_path"], "|", str(OUTPUT_REPORT_PATH))

print("=== OUTPUT HASH ===")
print({"parquet_sha256": write_manifest["parquet_sha256"], "csv_sha256": write_manifest["csv_sha256"]})

print("=== NEXT NOTEBOOK ===")
print("03CollectArticleBodies.ipynb")

=== ROW COUNTS ===
{'title_labels': 102}
=== NULL COUNTS ===
{'articleId': 0, 'rawBracketToken': 0, 'normalizedBracketToken': 0, 'normalizationMethod': 0, 'normalizationVersion': 0, 'isExcludedEditorialLabel': 0, 'extractionMethod': 0, 'extractionVersion': 0}
=== DUPLICATES ===
{'articleId': 0}
=== QUALITY METRICS ===
{'row_count_matches_expected': True, 'excluded_count': 1, 'aliased_count': 1, 'unique_raw_labels': 15, 'unique_normalized_labels': 14}
=== OUTPUT PATH ===
/home/sieg/projects-wsl/mbN_GUIDE_PY/mbN_GUIDE/data/20_processed/mbn/life/title_labels.parquet | /home/sieg/projects-wsl/mbN_GUIDE_PY/mbN_GUIDE/data/20_processed/mbn/life/title_labels.csv | /home/sieg/projects-wsl/mbN_GUIDE_PY/mbN_GUIDE/data/80_quality/label_normalization_report.json
=== OUTPUT HASH ===
{'parquet_sha256': '650c96d294c7b3e7d137c85738cde04abaf6abf9b3504bd60d4586b0c159975c', 'csv_sha256': '5d7afab39fe47ccfee31d1d8210d633d49fa15e28d69f0b3aef5bde3568d4b8a'}
=== NEXT NOTEBOOK ===
03CollectArticleBodies.ipynb
